In [3]:
import numpy as np
import pandas as pd
import torch

import sys
sys.path.append('../')
from utilities import binning_equal_q

In [4]:
def infer_fields(data, counts):
    
    fields = np.zeros([7,16])
    for pos in range(7):
        for color in range(16):
            fields[pos,color] = (counts*((data[:,pos] == color).astype('int'))).sum() 

    print(counts.sum(), fields.sum(1))

    fields7 = fields[6,:][fields[6,:] > 0]
    fields7 = torch.tensor(np.log(fields7 / fields7[0]))
    fields = fields[:6,:] / fields[:6,0][:,np.newaxis]
    fields = torch.tensor(np.log(fields))
    
    return fields, fields7

def sorted_log10p_vector_from_fields(fields, fields7):
    
    lengths = [torch.arange(16, dtype=torch.int8) for i in range(6)]
    lengths.append(torch.arange(11, dtype=torch.int8))
    all_seq = torch.cartesian_prod(*lengths)
    
    all_seq = all_seq.long()
    
    # generate p_vector
    p_vector = torch.zeros(11*16**6,dtype=torch.float32)

    for i in range(6):
        p_vector += fields[i,all_seq[:,i]]
        print(i)
    p_vector += fields7[all_seq[:,6]]
    
    p_vector = torch.exp(p_vector)
    
    Z = torch.exp(fields).sum(1).prod() * torch.exp(fields7).sum()
    
    p_vector /= Z
    print(p_vector.sum())
    
    return torch.log(p_vector).sort()[0] / np.log(10)

def log10q_vector_func(data, fields, fields7):
    
    # generate q_vector
    q_vector = torch.zeros(len(data))

    for i in range(6):
        q_vector += fields[i,data[:,i]]
        print(i)
    q_vector += fields7[data[:,6]]

    q_vector = torch.exp(q_vector)

    Z = torch.exp(fields).sum(1).prod() * torch.exp(fields7).sum()

    q_vector /= Z

    print(q_vector.sum())
    return np.log10(q_vector.numpy())

In [11]:
t = 4

In [12]:
data = pd.read_csv('../data/Dalkara.csv',index_col=0).query('T0 > 0')

transform_dict = {np.unique(data['a7'])[i] : i for i in range(11)}
data['a7'] = data['a7'].replace(transform_dict)

data = data.sort_values('T0', ascending=False)

np.random.seed(0)
data['sample'] = np.random.binomial(data['T0'], p=np.array([1./t]))
data = data.query('sample > 0')

counts = data['sample'].to_numpy()
data = data.iloc[:,:7].to_numpy()

In [13]:
fields, fields7 = infer_fields(data, counts)

3413805 [3413805. 3413805. 3413805. 3413805. 3413805. 3413805. 3413805.]


In [14]:
sorted_log10p_vector = sorted_log10p_vector_from_fields(fields, fields7)

0
1
2
3
4
5
tensor(1.0000)


In [15]:
log10q_vector = log10q_vector_func(data, fields, fields7)
argsort = np.argsort(log10q_vector)
sorted_log10q_vector = log10q_vector[argsort][::-1]
counts = counts[argsort][::-1]

0
1
2
3
4
5
tensor(0.7281)


In [16]:
bins=200
df_bins = binning_equal_q(sorted_log10p_vector, sorted_log10q_vector, counts, bins=bins, writefolder=False)#'results_DalkaraT0_IM', step=1000)
df_bins.to_csv('df_bins_Dalkara_IM_%dt.csv'%t)

0
tensor(184549376) tensor(184545312)
elements in the bin: 4064
nonzeros: 4063
1
tensor(184545312) tensor(184541249)
elements in the bin: 4063
nonzeros: 4063
2
tensor(184541249) tensor(184537186)
elements in the bin: 4063
nonzeros: 4063
3
tensor(184537186) tensor(184533123)
elements in the bin: 4063
nonzeros: 4063
4
tensor(184533123) tensor(184529060)
elements in the bin: 4063
nonzeros: 4063
5
tensor(184529060) tensor(184524997)
elements in the bin: 4063
nonzeros: 4063
6
tensor(184524997) tensor(184520930)
elements in the bin: 4067
nonzeros: 4063
7
tensor(184520930) tensor(184516864)
elements in the bin: 4066
nonzeros: 4063
8
tensor(184516864) tensor(184512789)
elements in the bin: 4075
nonzeros: 4063
9
tensor(184512789) tensor(184508713)
elements in the bin: 4076
nonzeros: 4063
10
tensor(184508713) tensor(184504634)
elements in the bin: 4079
nonzeros: 4063
11
tensor(184504634) tensor(184500548)
elements in the bin: 4086
nonzeros: 4063
12
tensor(184500548) tensor(184496453)
elements in

111
tensor(183649767) tensor(183629795)
elements in the bin: 19972
nonzeros: 4063
112
tensor(183629795) tensor(183609308)
elements in the bin: 20487
nonzeros: 4063
113
tensor(183609308) tensor(183588475)
elements in the bin: 20833
nonzeros: 4063
114
tensor(183588475) tensor(183567251)
elements in the bin: 21224
nonzeros: 4063
115
tensor(183567251) tensor(183545598)
elements in the bin: 21653
nonzeros: 4063
116
tensor(183545598) tensor(183522616)
elements in the bin: 22982
nonzeros: 4063
117
tensor(183522616) tensor(183499096)
elements in the bin: 23520
nonzeros: 4063
118
tensor(183499096) tensor(183475536)
elements in the bin: 23560
nonzeros: 4063
119
tensor(183475536) tensor(183451483)
elements in the bin: 24053
nonzeros: 4063
120
tensor(183451483) tensor(183426989)
elements in the bin: 24494
nonzeros: 4063
121
tensor(183426989) tensor(183400521)
elements in the bin: 26468
nonzeros: 4063
122
tensor(183400521) tensor(183373652)
elements in the bin: 26869
nonzeros: 4063
123
tensor(18337